# Problem Set 2 (Conceptual problems)
**Learning from data [TIF285], Chalmers, Fall 2026**  
*Last modified: 2026-09-28*

Total: 12 points.

<a id="toc"></a>
## Table of contents

- [Setup](#setup)
- [Problem 1: Maximum entropy reconstruction from moments (2 points)](#problem1)
- [Problem 2: Is there an annual modulation? (3 points)](#problem2)
- [Problem 3: Nuclear binding energies with a neural network (3 points)](#problem3)
- [Problem 4: Gaussian process regression (4 points)](#problem4)

<a id="setup"></a>
## Setup

Create the directories for data files and figures and import the modules used
throughout this notebook. Run this cell first, and re-run it after any kernel
restart.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

In [ ]:
import os

import numpy as np
import scipy.stats as stats
import scipy.optimize as optimize
from scipy.integrate import trapezoid
from scipy.special import logsumexp
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import sklearn.gaussian_process as gp_sklearn

# Data files are stored in
DATA_DIR = "DataFiles/"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Figures are stored in
FIGURES_DIR = "Figures/"

if not os.path.exists(FIGURES_DIR):
    os.makedirs(FIGURES_DIR)

rng = np.random.default_rng(seed=2026)
torch.manual_seed(2026);

<a id="problem1"></a>
## Problem 1: Maximum entropy reconstruction from moments (2 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

In many-body physics one often cannot compute a response function or a
spectral distribution directly, but its *moments* are accessible: sum rules
give $\int \omega^k \Phi(\omega)\, d\omega$ for a strength function $\Phi(\omega)$ for the first few $k$, and Lanczos-type
methods produce a handful of moments of a strength function at modest cost.
The same situation arises whenever a detector or a theory delivers a few
integrated numbers rather than the full distribution.

Here, we consider a normalized probability density $p(x)$ on the interval
$x \in [0,1]$, of which we know only the first $N$ moments

$$
\mu_k = \int_0^1 x^k p(x)\, dx, \qquad k = 1, 2, \ldots, N,
$$

together with the normalization $\mu_0 = 1$. Infinitely many densities share
these moments. The principle of maximum entropy tells us to pick the one that
is *least informative* given the constraints, i.e. the one that maximizes

$$
S[p] = -\int_0^1 p(x) \ln \frac{p(x)}{m(x)}\, dx,
$$

where we take the Lebesgue measure $m(x) = 1$ on the interval.

### (a) The maximum entropy density (1 point)

**Tasks (see also Yata)**
* Use the method of Lagrange multipliers to derive the form of the density that
  maximizes $S[p]$ under the $N+1$ constraints. Write down the constrained
  functional $Q[p]$ with multipliers $\lambda_0, \lambda_1, \ldots, \lambda_N$
  (one for the normalization and one per moment), set its functional derivative
  $\delta Q / \delta p(x)$ to zero, and show that the solution has the form

  $$
  p(x) = \exp\left( -\sum_{k=0}^{N} \lambda_k x^k \right),
  $$

  where the constant that appears when differentiating $p \ln p$ has been
  absorbed into $\lambda_0$. Write also the $N+1$ equations that determine the
  multipliers. Use the markdown cell below.
* Implement the density `maxent_pdf(x, lam)` and the function
  `moment_residuals(lam, moments, x)` that returns the vector of constraint
  violations $\int_0^1 x^k p(x)\,dx - \mu_k$ for $k = 0, \ldots, N$, evaluated by
  numerical quadrature on the grid `x` (use `scipy.integrate.trapezoid`). The
  roots of this vector function are the Lagrange multipliers we are after.

*Hint:* The functional derivative of $\int F(p(x))\,dx$ with respect to $p(x)$
is $F'(p(x))$, so you can treat the problem exactly like the finite-dimensional
one for a discrete set of probabilities $\{p_i\}$, with the sum over $i$
replaced by an integral over $x$.

**Your derivation here**

****************************
** Write your answer here **
****************************


In [ ]:
def maxent_pdf(x, lam):
    '''
    The maximum-entropy density p(x) = exp( - sum_k lam[k] x^k ).

    Args:
        x: array of positions, any shape
        lam: array-like of length N+1 with the Lagrange multipliers
            lam[0], ..., lam[N]

    Returns:
        p: array with the same shape as x
    '''
##################
# YOUR CODE HERE #
##################


def moment_residuals(lam, moments, x):
    '''
    Constraint violations of the maximum-entropy density for given multipliers.

    Args:
        lam: array-like of length N+1 with the Lagrange multipliers
        moments: array-like of length N+1 with the target moments
            (mu_0, mu_1, ..., mu_N), where mu_0 = 1
        x: one-dimensional grid on [0,1] used for the quadrature

    Returns:
        residuals: array of shape (N+1,) with
            int_0^1 x^k p(x) dx - mu_k  for k = 0, ..., N
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
x_grid = np.linspace(0., 1., 2001)

assert maxent_pdf(x_grid, [0.]).shape == x_grid.shape, 'maxent_pdf should preserve the shape of x'
assert np.allclose(maxent_pdf(x_grid, [0.]), 1.), 'With lam=[0] the density should be 1 everywhere'
assert np.allclose(maxent_pdf(x_grid, [0., 1.]), np.exp(-x_grid)), 'With lam=[0,1] the density should be exp(-x)'
assert np.allclose(maxent_pdf(x_grid, [np.log(2.), 0., 3.]), 0.5*np.exp(-3*x_grid**2)), \
    'Check the powers of x in the exponent'

res = moment_residuals([0.], [1.], x_grid)
assert res.shape == (1,), 'The residual vector should have one element per multiplier'
assert np.abs(res[0]) < 1e-8, 'The uniform density is normalized: the residual should vanish'
res = moment_residuals([0., 0.], [1., 0.5], x_grid)
assert np.allclose(res, [0., 0.], atol=1e-8), 'The uniform density has mean 0.5'
res = moment_residuals([0., 1.], [1., 0.5], x_grid)
assert np.allclose(res, [1. - np.exp(-1.) - 1., (1. - 2*np.exp(-1.)) - 0.5], atol=1e-6), \
    'Check the residuals for p(x) = exp(-x): the k-th element is int x^k p dx - mu_k'

### (b) Reconstruction from an increasing number of moments (1 point)

The file `PS2_Prob1_moments.txt` in the `DataFiles` directory contains the first
eight moments $\mu_1, \ldots, \mu_8$ of an unknown density. For comparison only,
the file `PS2_Prob1_true_pdf.txt` contains the true density on a grid. **Do not
use the true density anywhere in the reconstruction.**

**Tasks (see also Yata)**
* Determine the Lagrange multipliers for $N = 1, 2, 3, 4, 5, 6$ moments by
  finding the root of `moment_residuals` with `scipy.optimize.fsolve`. Start
  from $\lambda_k = 0$ for all $k$ (the uniform density). Store the solutions in
  the dictionary `lam_solutions` with the number of moments $N$ as key.
* Plot the six reconstructed densities together with the true density (one
  panel per $N$).
* Compute the entropy $S_N$ of each reconstruction and plot it as a function of
  $N$. How does it depend on $N$, and why must it behave like that?
* At which $N$ does the reconstruction first show that the true density has two
  peaks? Comment on how much information about the shape is contained in the
  lowest moments.

*Hint:* `fsolve(func, x0, args=(...))` passes the extra arguments on to
`func(x, *args)`. Check the residuals of the returned solution: `fsolve` does
not raise an error when it fails to converge.

In [ ]:
# Read the moments and the true density (the latter for comparison only)
moments_data = np.loadtxt(f'{DATA_DIR}PS2_Prob1_moments.txt')
x_true, p_true = np.loadtxt(f'{DATA_DIR}PS2_Prob1_true_pdf.txt', unpack=True)

print('Known moments:')
for k, mu_k in enumerate(moments_data, start=1):
    print(f'  mu_{k} = {mu_k:.6f}')

In [ ]:
# Solve for the Lagrange multipliers for N = 1, ..., 6 moments.
# Store the solutions in the dictionary lam_solutions[N]
lam_solutions = {}
entropies = {}

##################
# YOUR CODE HERE #
##################


In [ ]:
for N in range(1, 7):
    assert N in lam_solutions, f'lam_solutions is missing the key N={N}'
    assert len(lam_solutions[N]) == N+1, f'The solution for N={N} moments should have N+1 = {N+1} multipliers'
    moments = np.concatenate(([1.], moments_data[:N]))
    assert np.abs(moment_residuals(lam_solutions[N], moments, x_grid)).max() < 1e-5, \
        f'The multipliers for N={N} do not reproduce the moments'

In [ ]:
# Plot the reconstructions and the entropy as a function of N
##################
# YOUR CODE HERE #
##################


**Your comments here**

****************************
** Write your answer here **
****************************


<a id="problem2"></a>
## Problem 2: Is there an annual modulation? (3 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

Assume that you are analyzing a time series of data from an experiment that is
searching for an annual modulation on top of a constant background. For
example, an experiment that succeeds in detecting a flux of dark matter
particles reaching the Earth's surface would be expected to observe an annual
modulation of the signal, since the velocity of the Earth relative to the
galactic dark matter halo changes over the year. The kinematics also fixes the
*phase* of such a signal: the Earth's orbital velocity adds to the Sun's motion
through the halo around June 2, so the flux should be maximal then and minimal
half a year later.

The time series consists of $N = 60$ measurements $y_i$ at times $t_i$ (in
days, with $t = 0$ on January 1) that extend over a period of 5 years. The measurements are presented in
some re-scaled, dimensionless units. From calibration measurements it is known
that the experimental errors are normal distributed with a standard deviation
$\sigma = 5.0$.

Three data sets are provided in the files `PS2_Prob2_data0.txt`,
`PS2_Prob2_data1.txt` and `PS2_Prob2_data2.txt` (columns: $t_i$ in days, $y_i$).
Each set contains a constant background and might, or might not, include an
annual modulation.

In [ ]:
# The known experimental error, the length of a year (in days), and the predicted day of maximum
sigma = 5.0
T_year = 365.25
t0_signal = 152.5   # June 2, with t = 0 on January 1

# Read the three data sets
data_sets = []
for icase in range(3):
    t, y = np.loadtxt(f'{DATA_DIR}PS2_Prob2_data{icase}.txt', unpack=True)
    data_sets.append((t, y))
N = len(t)
print(f'{N} measurements per data set, spanning {t[-1]/T_year:.1f} years')

fig, axs = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
for icase, (t, y) in enumerate(data_sets):
    axs[icase].errorbar(t/T_year, y, yerr=sigma, fmt='o', ms=3)
    axs[icase].set_xlabel(r'$t$ (years)')
    axs[icase].set_title(f'data set {icase}')
axs[0].set_ylabel(r'$y$')
fig.tight_layout()

### (a) The chi-squared statistic and the null hypothesis (1 point)

Consider as null hypothesis $H_0$ the scenario that the signal is *constant*,
albeit with an unknown amplitude (the background).

**Tasks (see also Yata)**
* Implement the chi-squared statistic
  $$
  \chi^2 = \sum_{i=1}^N \frac{( y_i - \bar{y} )^2}{\sigma^2},
  $$
  via the function `chi2_statistic` below. The average value $\bar{y}$ is
  estimated from the mean of the samples $\{ y_i \}_{i=1}^{N}$. Compute the statistic
  for each of the three data sets.
* Implement the function `simulate_null_chi2` that generates $M$ hypothetical
  data sets (each with $N$ measurements) from the assumption that $H_0$ is
  true, and returns the $\chi^2$ statistic of each. The background should be
  set randomly for each realization (sample it from a uniform distribution in
  $[10, 200]$) and the noise should be drawn from a normal distribution with
  standard deviation $\sigma$. Use the random number generator passed as an
  argument.
* Generate $M = 10\,000$ null-hypothesis data sets. Which fraction of them gives a
  $\chi^2$ statistic that is *larger* than the value obtained for each of the
  three measured data sets? Store the three fractions in the list
  `null_fraction`.

In [ ]:
def chi2_statistic(y, dy):
    '''
    Chi-squared statistic for data with a known error and an unknown, constant background.

    Since the background is unknown, the sample mean is used as its estimate.

    Args:
        y: data, array of floats of shape (N,)
        dy: (float) fixed error, standard deviation of the normal distribution of errors

    Returns:
        chi2: (float) the chi-squared statistic
    '''
##################
# YOUR CODE HERE #
##################


def simulate_null_chi2(N, dy, M, rng, background_range=(10., 200.)):
    '''
    Chi-squared statistics of M simulated data sets generated under the null hypothesis
    (constant signal with a random background plus normal noise).

    Args:
        N: (int) number of measurements per data set
        dy: (float) standard deviation of the normal noise
        M: (int) number of simulated data sets
        rng: numpy random Generator used for all random numbers
        background_range: (lo, hi) range of the uniform distribution for the background

    Returns:
        chi2_null: array of shape (M,) with the chi-squared statistic of each simulated data set
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
assert isinstance(chi2_statistic(np.array([1., 2., 3.]), 1.), (np.floating, float)), 'chi2_statistic should return a float'
assert np.isclose(chi2_statistic(np.array([1., 2., 3.]), 1.), 2.), 'chi2 for [1,2,3] with dy=1 should be 2'
assert np.isclose(chi2_statistic(np.array([101., 102., 103.]), 2.), 0.5), 'The background should not matter; check the division by dy**2'

_chi2_null = simulate_null_chi2(60, 5.0, 2000, np.random.default_rng(1))
assert _chi2_null.shape == (2000,), 'simulate_null_chi2 should return one chi2 value per simulated data set'
assert np.all(_chi2_null >= 0.), 'chi2 cannot be negative'
assert np.abs(_chi2_null.mean() - 59.) < 3., 'Under the null hypothesis the mean chi2 should be close to N-1 = 59'
assert np.all(simulate_null_chi2(60, 5.0, 5, np.random.default_rng(7)) == simulate_null_chi2(60, 5.0, 5, np.random.default_rng(7))), \
    'Use the random number generator that is passed as an argument, so that results are reproducible'

In [ ]:
# Compute the chi2 statistic of the three data sets, simulate the null hypothesis, and compare.
# Store the chi2 statistic of the data in the list `chi2_from_data`, and the fractions of
# null-hypothesis data sets with a larger chi2 in the list `null_fraction`.
chi2_from_data = []
null_fraction = []
M = 10000

##################
# YOUR CODE HERE #
##################


In [ ]:
assert len(chi2_from_data) == 3 and len(null_fraction) == 3
assert np.isclose(chi2_from_data[0], chi2_statistic(data_sets[0][1], sigma))
assert np.all((np.array(null_fraction) >= 0.) & (np.array(null_fraction) <= 1.)), 'The fractions should be in [0,1]'
assert null_fraction[0] > null_fraction[2], 'Data set 2 should be less compatible with the null hypothesis than data set 0'

### (b) p-values and null hypothesis rejection (1 point)

**Tasks (see also Yata)**
* Under $H_0$, the statistic $\chi^2$ follows a chi-squared distribution. Which
  is the relevant number of degrees of freedom, given that the background is
  estimated from the data? Implement the function `p_value(chi2_value, dof)`
  that returns the $P$-value of an observed statistic, i.e. the probability
  under $H_0$ of obtaining a value at least as large as the observed one.
* Make a plot of the expected $\chi^2$ distribution (compare it also with the
  histogram of your simulated null-hypothesis data sets) and indicate the
  statistic obtained for the actual data in each case.
* Compute the $P$-value for each data set and store the values in the list
  `pvalue`. Does it agree with the fractions you found from the simulated
  null-hypothesis data sets?
* Assume that we have decided *beforehand* on a significance level of 5% for
  this test. Would you *reject the null hypothesis* for each particular case?
  Store your answers in the list of booleans `reject_null`.

In [ ]:
def p_value(chi2_value, dof):
    '''
    P-value of an observed chi-squared statistic.

    Args:
        chi2_value: (float) the observed statistic
        dof: (int) number of degrees of freedom

    Returns:
        p: (float) probability under the null hypothesis of a statistic >= chi2_value
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
assert np.isclose(p_value(0., 10), 1.), 'A statistic of zero has P-value 1'
assert np.isclose(p_value(stats.chi2(59).ppf(0.95), 59), 0.05), 'p_value should be the survival function of the chi2 distribution'
assert p_value(100., 59) < p_value(60., 59), 'A larger statistic must give a smaller P-value'

In [ ]:
# Compute the P-values, plot, and decide.
pvalue = []
reject_null = []
significance_level = 0.05

##################
# YOUR CODE HERE #
##################


In [ ]:
assert len(pvalue) == 3 and len(reject_null) == 3
assert all(isinstance(r, (bool, np.bool_)) for r in reject_null), 'reject_null should contain booleans'
assert np.allclose(pvalue, null_fraction, atol=0.02), 'The P-values should agree with the simulated fractions to within the Monte Carlo error'
assert reject_null[2] and not reject_null[0], 'Check your decisions for data sets 0 and 2'

### (c) Bayesian model comparison (1 point)

The $P$-value tells us how surprising the data are *if $H_0$ is true*. It says
nothing about the alternative. The Bayesian approach instead compares two
explicit models,

* $M_0$: $y_i = B + \varepsilon_i$,
* $M_1$: $y_i = B + A \cos\left(2\pi (t_i - t_0)/T_\mathrm{year}\right) + \varepsilon_i$,

with $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$, through the ratio of their
evidences (the Bayes factor). The phase is not a free parameter: the day of
maximum $t_0$ is predicted by the kinematics (June 2, i.e. $t_0 = 152.5$ days,
stored in `t0_signal` above), so $M_1$ has two parameters, $B$ and $A \geq 0$.
The Bayes factor is

$$
B_{10} = \frac{p(\mathcal{D}|M_1)}{p(\mathcal{D}|M_0)}, \qquad
p(\mathcal{D}|M) = \int p(\mathcal{D}|\boldsymbol{\theta}, M)\, p(\boldsymbol{\theta}|M)\, d\boldsymbol{\theta} .
$$

Use uniform priors: $B \sim \mathcal{U}(10, 200)$ in both models, and
$A \sim \mathcal{U}(0, A_\mathrm{max})$ in $M_1$.

**Evidence on a grid.** The parameter spaces are small enough that the
evidence integrals can be evaluated on a grid. With grid cells of volume
$\Delta$ the integral becomes a sum,

$$
p(\mathcal{D}|M) \approx \sum_{j} p(\mathcal{D}|\boldsymbol{\theta}_j, M)\, p(\boldsymbol{\theta}_j|M)\, \Delta
= \frac{\Delta}{V} \sum_{j} p(\mathcal{D}|\boldsymbol{\theta}_j, M),
$$

where the second form uses that a normalized uniform prior equals $1/V$ inside
its range, with $V$ the volume of the range. Taking the logarithm,

$$
\ln p(\mathcal{D}|M) \approx \underbrace{\ln \sum_j e^{\ln p(\mathcal{D}|\boldsymbol{\theta}_j, M)}}_{\texttt{logsumexp}} - \ln V + \ln \Delta ,
$$

which is evaluated with `scipy.special.logsumexp` to avoid underflow. We use
grids whose points are the *centres* of $n$ equal cells covering the prior
range (`midpoint_grid` below), so that the spacing is `grid[1] - grid[0]`, the
prior range has length $n \times$ spacing, and $V$ and $\Delta$ are products
of these over the parameters.

The functions below are given: a Gaussian log likelihood
`log_likelihood_grid` that evaluates many parameter combinations at once (the
last axis of the model array runs over the $N$ data points, the leading axes
over parameter values), the model function `model_M0`, and `log_evidence_M0`,
which implements the recipe for the constant model with one line per term.
Read them carefully: your task is to do the same for $M_1$.

**Tasks (see also Yata)**
* Complete `model_M1(t, B, A)` (see `model_M0` for the broadcasting
  convention) and `log_evidence_M1(t, y, dy, B_grid, A_grid)`: evaluate the
  model for all $(B, A)$ pairs and all $N$ measurement times at once, which
  gives a log likelihood of shape `(nB, nA)`, and then follow the recipe of
  `log_evidence_M0` with the prior volume and the cell volume of the
  two-dimensional grid.
* Complete the loop in the cell below that computes $\ln B_{10}$ for the three
  data sets and for $A_\mathrm{max} \in \{5, 10, 20, 50, 100\}$, stored in the
  array `lnB10_vs_Amax` of shape `(3, 5)` (one row per data set, one column
  per $A_\mathrm{max}$). Interpret the results with the scale below, using the
  column for $A_\mathrm{max} = 10$ as the reference case, and compare with the
  $P$-values from (b). Would you announce a discovery in any of the three
  cases? Explain the trend with $A_\mathrm{max}$: for which data set does the
  *conclusion* depend on the prior?

The strength of evidence is conventionally read off a scale of the type
introduced by Jeffreys. We use the version of
[Trotta, Contemp. Phys. 49, 71 (2008), Table 1](https://arxiv.org/abs/0803.4089),
which the course literature also refers to:

| $\lvert \ln B_{10} \rvert$ | $\lvert B_{10} \rvert$ | Strength of evidence |
|---|---|---|
| $< 1$ | $< 3$ | inconclusive |
| 1 -- 2.5 | 3 -- 12 | weak |
| 2.5 -- 5 | 12 -- 150 | moderate |
| $> 5$ | $> 150$ | strong |

A positive $\ln B_{10}$ favours $M_1$ (a modulation), a negative one favours
$M_0$ (no modulation). The labels are conventions, not theorems; other
authors (e.g. Kass and Raftery 1995) draw the boundaries slightly differently.

In [ ]:
def midpoint_grid(lo, hi, n):
    '''Centres of n equal cells covering the interval [lo, hi].'''
    return lo + (np.arange(n) + 0.5) * (hi - lo) / n


def log_likelihood_grid(y, y_model, dy):
    '''
    Gaussian log likelihood of the data y for model predictions evaluated at many parameter values.

    Args:
        y: array (N,) of measurements
        y_model: array (..., N) of model predictions. The last axis runs over the N
            measurement times; the leading axes (if any) run over parameter values,
            e.g. (nB, N) for the constant model or (nB, nA, N) for the modulation model.
        dy: (float) standard deviation of the normal errors

    Returns:
        array with the shape of the leading axes, e.g. (nB,) or (nB, nA): the log likelihood
        (including the normalization) for each parameter combination
    '''
    N = len(y)
    return -0.5*np.sum((y - y_model)**2, axis=-1)/dy**2 - N*np.log(dy*np.sqrt(2*np.pi))


def model_M0(t, B):
    '''
    The constant model. The arguments may be arrays that broadcast against each other,
    e.g. t of shape (N,) and B of shape (nB, 1) give an array of shape (nB, N).
    '''
    return B + 0.*t


def log_evidence_M0(t, y, dy, B_grid):
    '''
    Log evidence of the constant model M0 with a uniform prior on B (given; a worked example).

    Args:
        t: array (N,) of measurement times (not used by this model)
        y: array (N,) of measurements
        dy: (float) standard deviation of the normal errors
        B_grid: array (nB,) of cell-centre grid points covering the prior range of B

    Returns:
        logZ: (float) natural logarithm of the evidence p(D|M0)
    '''
    dB = B_grid[1] - B_grid[0]                            # grid spacing = cell volume
    log_volume = np.log(len(B_grid) * dB)                 # ln V of the uniform prior
    y_model = model_M0(t, B_grid[:, np.newaxis])          # shape (nB, N): one row per B, one column per measurement
    log_like = log_likelihood_grid(y, y_model, dy)        # shape (nB,)
    return logsumexp(log_like) - log_volume + np.log(dB)

In [ ]:
def model_M1(t, B, A):
    '''
    The modulation model y = B + A cos(2 pi (t - t0_signal) / T_year), with the phase fixed
    at the predicted day of maximum t0_signal.
    As in model_M0, the arguments may be arrays that broadcast against each other.
    '''
##################
# YOUR CODE HERE #
##################


def log_evidence_M1(t, y, dy, B_grid, A_grid):
    '''
    Log evidence of the modulation model M1 with uniform priors on B and A.

    Args:
        t: array (N,) of measurement times in days
        y: array (N,) of measurements
        dy: (float) standard deviation of the normal errors
        B_grid: array (nB,) of cell-centre grid points covering the prior range of B
        A_grid: array (nA,) of cell-centre grid points covering the prior range of A

    Returns:
        logZ: (float) natural logarithm of the evidence p(D|M1)
    '''
    # Hint: model_M1(t, B_grid[:, np.newaxis, np.newaxis], A_grid[np.newaxis, :, np.newaxis])
    # has shape (nB, nA, N); the corresponding log likelihood has shape (nB, nA).
##################
# YOUR CODE HERE #
##################


In [ ]:
B_grid = midpoint_grid(10., 200., 380)

_t, _y = data_sets[0]
assert np.isclose(model_M1(t0_signal, 100., 3.), 103.), 'At the predicted day of maximum the model should be B + A'
assert np.isclose(model_M1(t0_signal + T_year/2, 100., 3.), 97.), 'Half a year later the model should be B - A'
assert model_M1(_t, midpoint_grid(10., 200., 4)[:, np.newaxis, np.newaxis],
                midpoint_grid(0., 10., 3)[np.newaxis, :, np.newaxis]).shape == (4, 3, len(_t)), \
    'model_M1 should broadcast its arguments: shape (nB, nA, N) for the arguments shown in the hint'

_logZ0 = log_evidence_M0(_t, _y, sigma, B_grid)
# The evidence integral over B can be done analytically for the constant model:
_N = len(_y)
_analytic = (-0.5*np.sum((_y - _y.mean())**2)/sigma**2 - _N*np.log(sigma*np.sqrt(2*np.pi))
             + 0.5*np.log(2*np.pi*sigma**2/_N) - np.log(190.))
assert np.abs(_logZ0 - _analytic) < 1e-3, 'log_evidence_M0 does not agree with the analytic integral over B (this is the given function; did you change log_likelihood_grid?)'
# With A_max -> 0 the modulation model reduces to the constant model
_logZ1 = log_evidence_M1(_t, _y, sigma, B_grid, midpoint_grid(0., 1e-6, 2))
assert isinstance(_logZ1, (np.floating, float)), 'log_evidence_M1 should return a float'
assert np.abs(_logZ1 - _logZ0) < 1e-4, 'For a vanishing amplitude range, M1 should reduce to M0. Check the prior volume and the cell volume of M1.'

In [ ]:
# Compute ln B10 for the three data sets and the five prior ranges A_max.
# Store the values in the array lnB10_vs_Amax: row = data set, column = A_max.
A_max_list = [5., 10., 20., 50., 100.]
lnB10_vs_Amax = np.zeros((3, len(A_max_list)))
for icase, (t, y) in enumerate(data_sets):
    logZ0 = log_evidence_M0(t, y, sigma, B_grid)
    # Loop over A_max_list: build the A grid (use 200 cells), compute ln B10, and store it.
##################
# YOUR CODE HERE #
##################

i_ref = A_max_list.index(10.)   # the reference column, A_max = 10
for icase in range(3):
    print(f'Data set {icase}: P-value = {pvalue[icase]:.4f},  ln B10 (A_max=10) = {lnB10_vs_Amax[icase, i_ref]:6.2f}')

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
for icase in range(3):
    ax.plot(A_max_list, lnB10_vs_Amax[icase], 'o-', label=f'data set {icase}')
ax.axhline(0., color='k', ls=':')
ax.set_xscale('log')
ax.set_xlabel(r'$A_\mathrm{max}$')
ax.set_ylabel(r'$\ln B_{10}$')
ax.legend()
fig.tight_layout()

In [ ]:
assert lnB10_vs_Amax.shape == (3, 5) and np.all(lnB10_vs_Amax != 0.), 'lnB10_vs_Amax should be filled for all data sets and all A_max'
assert np.all(np.diff(lnB10_vs_Amax[:, 1:], axis=1) < 0.), 'ln B10 should decrease with A_max once the prior range exceeds the amplitudes the data support'
assert lnB10_vs_Amax[2, 1] > 5., 'Data set 2 should give strong evidence for a modulation (A_max = 10)'
assert lnB10_vs_Amax[0, 1] < 0., 'Data set 0 should favour the constant model (A_max = 10)'

**Your comments here**

****************************
** Write your answer here **
****************************


<a id="problem3"></a>
## Problem 3: Nuclear binding energies with a neural network (3 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

The binding energy $B(N, Z)$ of a nucleus with $N$ neutrons and $Z$ protons is
the energy required to separate it into its constituents. It is one of the
most precisely measured quantities in nuclear physics, known for about 2500
nuclei, and at the same time one of the hardest to predict for nuclei that have
not yet been measured. The file `PS2_Prob3_binding_energies.txt` in the
`DataFiles` directory contains $Z$, $N$, $A = N+Z$, $B$ (in MeV), $B/A$ and
the experimental uncertainty (in keV) for 2452 nuclei with $Z \geq 8$ from the
2020 Atomic Mass Evaluation. Take a look at the header of the file.

The classic model is the **liquid-drop** (semi-empirical mass) formula,

$$
B_\mathrm{LD}(N, Z) = a_V A - a_S A^{2/3} - a_C \frac{Z(Z-1)}{A^{1/3}} - a_A \frac{(N-Z)^2}{A} + a_P \frac{\delta(N,Z)}{A^{1/2}},
$$

with $\delta = +1$ for even-even nuclei (both $N$ and $Z$ even), $\delta = -1$
for odd-odd nuclei and $\delta = 0$ otherwise. The five terms describe volume,
surface, Coulomb, asymmetry and pairing energies, and the model is *linear* in
its five parameters $(a_V, a_S, a_C, a_A, a_P)$.

In this problem you will fit the liquid-drop model, train a neural network on
the same data, and then ask the question that matters for a physicist: what
happens when the model is asked about nuclei it has never seen?

In [ ]:
# Read the data
Z_all, N_all, A_all, B_all, BA_all, dB_all = np.loadtxt(f'{DATA_DIR}PS2_Prob3_binding_energies.txt', unpack=True)
Z_all = Z_all.astype(int)
N_all = N_all.astype(int)
A_all = A_all.astype(int)
print(f'{len(Z_all)} nuclei, Z = {Z_all.min()}..{Z_all.max()}, N = {N_all.min()}..{N_all.max()}, A = {A_all.min()}..{A_all.max()}')

fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
sc = ax.scatter(N_all, Z_all, c=BA_all, s=6, cmap='viridis')
ax.set_xlabel(r'$N$')
ax.set_ylabel(r'$Z$')
fig.colorbar(sc, label=r'$B/A$ (MeV)')
fig.tight_layout()

### (a) The liquid-drop model (1 point)

**Tasks (see also Yata)**
* Implement `liquid_drop_features(Z, N)` that returns the design matrix
  $\mathbf{X}$ of shape `(n, 5)` whose columns are the five terms of the
  liquid-drop formula (with their signs, so that
  $B_\mathrm{LD} = \mathbf{X} \boldsymbol{a}$ with
  $\boldsymbol{a} = (a_V, a_S, a_C, a_A, a_P)$, all positive).
* Implement `fit_liquid_drop(Z, N, B)` that returns the least-squares estimate
  of the five parameters (use `numpy.linalg.lstsq`).
* Fit the model to all nuclei, print the parameters, and compute the root-mean-square
  (RMS) residual $\sqrt{\langle (B - B_\mathrm{LD})^2 \rangle}$ in MeV.
  Store the residuals in the array `residual_LD`.
* Plot the residual $B - B_\mathrm{LD}$ as a function of $N$ (and as a
  function of $Z$). Where are the largest deviations, and what do you think
  they are due to? (Look up "magic numbers" if the pattern does not ring a
  bell.)

In [ ]:
def liquid_drop_features(Z, N):
    '''
    Design matrix of the liquid-drop formula.

    Args:
        Z: array (n,) of proton numbers
        N: array (n,) of neutron numbers

    Returns:
        X: array (n, 5) with columns
            A,  -A^(2/3),  -Z(Z-1)/A^(1/3),  -(N-Z)^2/A,  delta/A^(1/2)
        where delta = +1 (even-even), -1 (odd-odd), 0 (odd A)
    '''
##################
# YOUR CODE HERE #
##################


def fit_liquid_drop(Z, N, B):
    '''
    Least-squares fit of the liquid-drop parameters.

    Args:
        Z, N: arrays (n,) of proton and neutron numbers
        B: array (n,) of binding energies in MeV

    Returns:
        a: array (5,) with the parameters (a_V, a_S, a_C, a_A, a_P) in MeV
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
_X = liquid_drop_features(np.array([26, 50, 8, 7]), np.array([30, 82, 8, 7]))
assert _X.shape == (4, 5), 'The design matrix should have shape (n, 5)'
assert np.allclose(_X[:, 0], [56, 132, 16, 14]), 'The first column should be A'
assert np.allclose(_X[:, 4], [1/np.sqrt(56), 1/np.sqrt(132), 1/np.sqrt(16), -1/np.sqrt(14)]), \
    'Check the pairing term: +1 for even-even, -1 for odd-odd, 0 for odd A'
assert np.isclose(_X[0, 2], -26*25/56**(1/3)), 'Check the Coulomb term'
assert np.isclose(_X[1, 3], -(82-50)**2/132), 'Check the asymmetry term'

_a = fit_liquid_drop(Z_all, N_all, B_all)
assert _a.shape == (5,), 'fit_liquid_drop should return five parameters'
assert np.all(_a > 0), 'All liquid-drop parameters should come out positive with the sign convention of the design matrix'
assert 14. < _a[0] < 17. and 0.6 < _a[2] < 0.8, 'The volume and Coulomb coefficients are far from their textbook values (~15.7 and ~0.71 MeV)'

In [ ]:
# Fit the liquid-drop model to all nuclei, print the parameters and the RMS residual, and plot the residuals.
# Store the residuals B - B_LD in the array `residual_LD`.
##################
# YOUR CODE HERE #
##################


In [ ]:
assert residual_LD.shape == B_all.shape
assert np.sqrt(np.mean(residual_LD**2)) < 3.5, 'The RMS residual of the liquid-drop fit should be about 3 MeV'
assert np.abs(np.mean(residual_LD)) < 0.1, 'The least-squares residuals should have (nearly) zero mean'

**Your comments here**

****************************
** Write your answer here **
****************************


### (b) A neural network for the binding energy per nucleon (1 point)

We now let a neural network learn $B/A$ as a function of $(Z, N)$ from the
data, with no physics put in by hand. The data preparation is given in the
cell below: the nuclei are split at random into 80% training and 20%
validation data, and both the inputs $(Z, N)$ and the target $B/A$ are
standardized (mean 0, standard deviation 1) using the mean and standard
deviation of the *training* data, before being converted to `torch` tensors.
Standardizing is important for the optimization: raw inputs of order 100 and
targets of order 8 MeV make the loss landscape badly scaled.

**Tasks (see also Yata)**
* Complete the class `BindingEnergyNet`: a fully connected network with two
  inputs, two hidden layers with `n_hidden` nodes each and `tanh` activations
  (the target is a smooth function, for which a smooth activation works well),
  and a single linear output. The demonstration *Feed-forward neural network
  for a function in PyTorch* in the lecture notes has a network of this type.
* Complete the function `train_model` that trains a model with full-batch
  gradient descent using the `Adam` optimizer and the mean-squared-error loss,
  and returns the training and validation loss after every epoch. The training
  loop is also in the demonstration; the only addition is to evaluate the loss
  on the validation data after each step (do that under `torch.no_grad()`).
* Run the given cell that trains a network with `n_hidden=64`, plots the two
  loss curves and compares the RMS error of the predicted *total* binding
  energy $B = A \times (B/A)_\mathrm{pred}$ on the validation nuclei with the
  RMS residual of the liquid-drop model on the same nuclei. Comment on the
  result.

*Hints:* the target tensors have shape `(n, 1)`, so the network output must
have that shape too. A `Linear(n_in, n_out)` layer has `n_in*n_out + n_out`
parameters; check that `sum(p.numel() for p in model.parameters())` gives what
you expect.

In [ ]:
# Given: data preparation for the network.
inputs_all = np.column_stack([Z_all, N_all]).astype(float)

def standardize(x, mean, std):
    '''Standardize x with the given mean and standard deviation (arrays broadcast).'''
    return (x - mean) / std

def to_tensor(a):
    '''NumPy array -> float32 torch tensor.'''
    return torch.tensor(a, dtype=torch.float32)

def prepare_data(target_all, train_mask):
    '''
    Split, standardize with the training statistics, and convert to tensors.

    Args:
        target_all: array (n,) with the target for all nuclei (e.g. B/A)
        train_mask: boolean array (n,), True for the training nuclei

    Returns:
        x_train, y_train, x_val, y_val: tensors of shapes (n_train, 2), (n_train, 1), (n_val, 2), (n_val, 1)
        stats: dict with the training means and standard deviations, needed to standardize new
               inputs and to transform predictions back to physical units
    '''
    x_mean, x_std = inputs_all[train_mask].mean(axis=0), inputs_all[train_mask].std(axis=0)
    y_mean, y_std = target_all[train_mask].mean(), target_all[train_mask].std()
    x_train = to_tensor(standardize(inputs_all[train_mask], x_mean, x_std))
    x_val = to_tensor(standardize(inputs_all[~train_mask], x_mean, x_std))
    y_train = to_tensor(standardize(target_all[train_mask], y_mean, y_std)).reshape(-1, 1)
    y_val = to_tensor(standardize(target_all[~train_mask], y_mean, y_std)).reshape(-1, 1)
    stats_dict = dict(x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std)
    return x_train, y_train, x_val, y_val, stats_dict

def predict_all(model, stats_dict):
    '''Predictions of a trained model for all nuclei, transformed back to physical units. Array (n,).'''
    model.eval()
    with torch.no_grad():
        pred = model(to_tensor(standardize(inputs_all, stats_dict['x_mean'], stats_dict['x_std']))).numpy()[:, 0]
    return pred * stats_dict['y_std'] + stats_dict['y_mean']

# Random 80/20 split
train_mask = rng.random(len(Z_all)) < 0.8
val_mask = ~train_mask
x_train, y_train, x_val, y_val, stats_BA = prepare_data(BA_all, train_mask)
print(f'{train_mask.sum()} training nuclei, {val_mask.sum()} validation nuclei')
print(f'x_train: {tuple(x_train.shape)}, y_train: {tuple(y_train.shape)}, dtype {x_train.dtype}')

In [ ]:
class BindingEnergyNet(nn.Module):
    '''
    Fully connected network (Z, N) -> B/A with two hidden tanh layers.
    '''
    def __init__(self, n_hidden=64):
        super().__init__()
##################
# YOUR CODE HERE #
##################

    def forward(self, x):
        '''
        Args:
            x: tensor of shape (n, 2) with standardized (Z, N)
        Returns:
            tensor of shape (n, 1)
        '''
##################
# YOUR CODE HERE #
##################


def train_model(model, x_train, y_train, x_val, y_val, epochs=3000, lr=5e-3):
    '''
    Full-batch training with Adam and the mean-squared-error loss.

    Args:
        model: an nn.Module
        x_train, y_train: training inputs (n_train, 2) and targets (n_train, 1), tensors
        x_val, y_val: validation inputs and targets, tensors
        epochs: number of epochs
        lr: learning rate

    Returns:
        history: dict with arrays 'train_loss' and 'val_loss' of length `epochs`
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
_model = BindingEnergyNet(n_hidden=8)
assert sum(p.numel() for p in _model.parameters()) == 2*8+8 + 8*8+8 + 8+1, \
    'The number of parameters does not match two hidden layers of width n_hidden'
assert _model(torch.zeros(5, 2)).shape == (5, 1), 'The output should have shape (n, 1)'
assert any(isinstance(m, nn.Tanh) for m in _model.modules()), 'Use tanh activations'

_x = torch.linspace(-1, 1, 20).reshape(-1, 1).repeat(1, 2)
_y = _x[:, :1]**2
_hist = train_model(_model, _x, _y, _x, _y, epochs=50, lr=1e-2)
assert set(_hist) >= {'train_loss', 'val_loss'} and len(_hist['train_loss']) == 50, \
    'train_model should return the losses for every epoch'
assert _hist['train_loss'][-1] < _hist['train_loss'][0], 'The training loss should decrease'

In [ ]:
# Given: train the network, plot the loss curves and compare with the liquid-drop model on the validation nuclei.
torch.manual_seed(2026)
model = BindingEnergyNet(n_hidden=64)
print(f'{sum(p.numel() for p in model.parameters())} trainable parameters')
history = train_model(model, x_train, y_train, x_val, y_val, epochs=4000, lr=5e-3)

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
ax.plot(history['train_loss'], label='training')
ax.plot(history['val_loss'], label='validation')
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss (standardized units)')
ax.legend()
fig.tight_layout()

B_pred_NN = predict_all(model, stats_BA) * A_all
rms_NN_val = np.sqrt(np.mean((B_all - B_pred_NN)[val_mask]**2))
rms_LD_val = np.sqrt(np.mean(residual_LD[val_mask]**2))
rms_NN_train = np.sqrt(np.mean((B_all - B_pred_NN)[train_mask]**2))
print(f'RMS error of B on the validation nuclei: network {rms_NN_val:.2f} MeV, liquid drop {rms_LD_val:.2f} MeV')
print(f'RMS error of B on the training nuclei:   network {rms_NN_train:.2f} MeV')

In [ ]:
assert rms_NN_val < rms_LD_val, 'With a random split, the network should beat the liquid-drop model on the validation nuclei'
assert rms_NN_val < 2.5, 'The network should reach an RMS error below 2.5 MeV on the validation nuclei; check the network and the training loop'

**Your comments here**

****************************
** Write your answer here **
****************************


### (c) Interpolation versus extrapolation (1 point)

A random split puts every validation nucleus next to training nuclei in the
$(N, Z)$ plane. That measures *interpolation*. The nuclei we would actually like
to predict are the ones beyond the measured region, and there the situation is
different.

The function `train_and_predict` below wraps the steps of (b) -- data
preparation, training and prediction for all nuclei -- for an arbitrary target
and an arbitrary training set, so that you can design the experiment yourself.

**Tasks (see also Yata)**
* Train on all nuclei with $A \leq 180$ and test on the heavy nuclei with
  $A > 180$. Build the boolean training mask, fit the liquid-drop model to the
  *training* nuclei only, and train two networks with `train_and_predict`: one
  on $B/A$ as in (b), and one on the residual of the liquid-drop model per
  nucleon, $(B - B_\mathrm{LD})/A$.
* Compute the RMS error of the total binding energy $B$ on the heavy nuclei
  for three predictions: the pure network, the liquid-drop model, and the
  hybrid $B_\mathrm{LD} + A \times \mathrm{NN}$. Store them in `rms_NN_heavy`,
  `rms_LD_heavy` and `rms_hybrid_heavy`.
* Plot the error $B - B_\mathrm{pred}$ of the pure network as a function of
  $A$ for all nuclei, marking the boundary of the training region. Where does
  the error start to grow? The given cell then plots $B/A$ along the $Z = 82$
  isotopic chain (all of which has $A > 180$) for the data and the three models.
* Summarize the RMS errors in a small table and comment. Why does the pure
  network fail where it was excellent in (b)? What is the advantage of letting
  the network learn a correction to a physical model rather than the full
  function? Which of the three models would you use to predict a nucleus with
  $A = 300$, and how far would you trust it?

In [ ]:
# Given: wrapper around the steps of (b). Each call takes 10-30 seconds.
def train_and_predict(target_all, train_mask, epochs=4000, lr=5e-3, seed=2026):
    '''
    Prepare the data, train a BindingEnergyNet on the nuclei selected by train_mask,
    and return the predictions for ALL nuclei in physical units (array (n,)).

    Args:
        target_all: array (n,) with the target for all nuclei, e.g. BA_all
        train_mask: boolean array (n,), True for the training nuclei
    '''
    x_tr, y_tr, x_va, y_va, stats_dict = prepare_data(target_all, train_mask)
    torch.manual_seed(seed)
    net = BindingEnergyNet(n_hidden=64)
    train_model(net, x_tr, y_tr, x_va, y_va, epochs=epochs, lr=lr)
    return predict_all(net, stats_dict)

In [ ]:
# Extrapolation experiment with the boundary at A = 180.
# Store the RMS errors of B on the heavy nuclei in rms_NN_heavy, rms_LD_heavy and rms_hybrid_heavy,
# and the three predictions for all nuclei in B_pred_NN_heavy, B_LD_light and B_pred_hybrid (used by the plot below).
##################
# YOUR CODE HERE #
##################


In [ ]:
_heavy = A_all > 180
assert np.isclose(rms_LD_heavy, np.sqrt(np.mean((B_all - B_LD_light)[_heavy]**2))), \
    'rms_LD_heavy should be the RMS error of the liquid-drop model on the A > 180 nuclei only'
assert rms_NN_heavy > rms_LD_heavy, 'In extrapolation the pure network is expected to do worse than the liquid-drop model'
assert rms_hybrid_heavy < rms_LD_heavy, 'The hybrid model should improve on the liquid-drop model for the heavy nuclei'
assert np.sqrt(np.mean((B_all - B_LD_light)[~_heavy]**2)) < 3.5, 'B_LD_light should be the liquid-drop model fitted to the training nuclei'

In [ ]:
# Given: the Pb (Z = 82) isotopic chain
chain = Z_all == 82
order = np.argsort(N_all[chain])
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.2))
ax.plot(N_all[chain][order], BA_all[chain][order], 'ko', ms=4, label='data')
ax.plot(N_all[chain][order], B_pred_NN_heavy[chain][order]/A_all[chain][order], 'C0-', label='network on $B/A$')
ax.plot(N_all[chain][order], B_LD_light[chain][order]/A_all[chain][order], 'C1--', label='liquid drop')
ax.plot(N_all[chain][order], B_pred_hybrid[chain][order]/A_all[chain][order], 'C2-', label='hybrid')
ax.set_xlabel(r'$N$')
ax.set_ylabel(r'$B/A$ (MeV)')
ax.set_title(r'$Z = 82$ isotopic chain (all $A > 180$: none in the training data)')
ax.legend()
fig.tight_layout()

**Your comments here**

****************************
** Write your answer here **
****************************


<a id="problem4"></a>
## Problem 4: Gaussian process regression (4 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

In the following we will perform Gaussian process (GP) regression and explore
* the prior prediction of a GP regression model with fixed hyperparameters,
* the posterior prediction of a GP regression model with fixed hyperparameters,
  but conditioned on training data,
* the dependence on the choice of kernel hyperparameters,
* the optimization of the hyperparameters by maximization of the (log) marginal
  likelihood, and
* what the GP does when it is asked to predict outside the data.

The data comes from problem 3: the file `PS2_Prob4_Sn_residuals.txt` contains
the residual $y = B - B_\mathrm{LD}$ (in MeV) of the liquid-drop model along the
tin ($Z = 50$) isotopic chain as a function of the neutron number $x = N$. The
liquid-drop parameters used are given in the header of the file. The residual
is a smooth function of $N$ with a peak at the magic number $N = 82$, exactly
the kind of structure that the liquid-drop model does not know about. Both the
input $x$ and the target $y$ are one-dimensional.

We will use the nuclei with $N \leq 80$ as training data and hold out the five
nuclei with $N = 81, \ldots, 85$ for the final part.

The GP demonstration notebook in the lecture notes (*Demo: Gaussian
processes*, sec. 32.4) contains the RBF kernel, the sampling of a GP prior and
the `sklearn` set-up that you will need in (a)--(c); the formulas for the GP
posterior are in secs. 32.2--32.3.

In [ ]:
# Read the data and split it
x_Sn, y_Sn = np.loadtxt(f'{DATA_DIR}PS2_Prob4_Sn_residuals.txt', unpack=True)
train = x_Sn <= 80
X_train = x_Sn[train].reshape(-1, 1)
Y_train = y_Sn[train].reshape(-1, 1)
X_test = x_Sn[~train].reshape(-1, 1)
Y_test = y_Sn[~train].reshape(-1, 1)
print(f'{len(X_train)} training points (N = {int(X_train.min())}..{int(X_train.max())}), '
      f'{len(X_test)} test points (N = {int(X_test.min())}..{int(X_test.max())})')

fig, ax = plt.subplots(1, 1, figsize=(6, 3.8))
ax.plot(X_train, Y_train, 'ko', label='training data')
ax.plot(X_test, Y_test, 'rs', mfc='none', label='held out')
ax.set_xlabel(r'$N$')
ax.set_ylabel(r'$B - B_\mathrm{LD}$ (MeV)')
ax.legend()
fig.tight_layout()

### (a) The kernel and the prior GP (1 point)

We model the data as $y_i = f(x_i) + \varepsilon_i$, where $f$ is a Gaussian
process with the RBF kernel

$$
k(x_i, x_j) = \sigma_f^2 \exp\left(-\frac{(x_i - x_j)^2}{2 \ell^2}\right)
$$

and $\varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)$ is independent white noise. The
correlation length $\ell$ controls the smoothness of the GP samples and
$\sigma_f$ the vertical variation. The kernel is stationary since it depends
only on the distance $x_i - x_j$. The noise describes the part of the data that
we do not try to model as a smooth function; here it stands for the fine
structure of the residual (e.g. odd-even effects that the pairing term does not
fully capture) and we fix it at $\sigma_n = 0.2$ MeV.

The covariance of two *observations* is then

$$
\mathrm{cov}(y_i, y_j) = k(x_i, x_j) + \sigma_n^2 \delta_{ij},
$$

where the Kronecker delta refers to the observation indices, not to the
positions: two different measurements have independent noise even if they are
taken at the same $x$. In matrix form, the covariance of the training data is
$C(\mathbf{X}, \mathbf{X}) = K(\mathbf{X}, \mathbf{X}) + \sigma_n^2 \mathbb{1}$,
where $K$ collects the kernel values. We will therefore implement only the
kernel matrix, for two possibly different sets of positions $\mathbf{X}$
(length $N$) and $\mathbf{X}'$ (length $M$), giving an $N \times M$ matrix, and
add $\sigma_n^2 \mathbb{1}$ explicitly where the model says so.

**Tasks (see also Yata)**
* Complete the function `rbf_kernel` below.
* Complete the utility function `gp_plot` to draw random samples from the GP
  and plot them together with the mean and the credible band (the latter are
  already implemented).
* Show the prior over functions with mean zero and a covariance obtained with
  $\sigma_f = 3$ MeV and $\ell = 5$ on the interval $N \in [50, 90]$. Since we
  are interested in what *measurements* would look like, add the noise
  variance $\sigma_n^2 \mathbb{1}$ to the covariance (`np.eye`). Use `gp_plot` to
  show the mean, the 95% band and 5 samples.

In [ ]:
def rbf_kernel(X1, X2, sigma_f, ell):
    '''
    RBF (squared exponential) kernel matrix between two sets of one-dimensional positions.

    Args:
        X1: array of shape (N, 1) (or (N,)) of positions
        X2: array of shape (M, 1) (or (M,)) of positions
        sigma_f: (float) signal standard deviation
        ell: (float) correlation length

    Returns:
        K: array of shape (N, M) with K[i,j] = sigma_f^2 exp(-(x1_i - x2_j)^2 / (2 ell^2))
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
_X1 = np.array([[0.], [1.], [2.]])
_X2 = np.array([[1.], [3.]])
_K = rbf_kernel(_X1, _X2, sigma_f=2., ell=1.)
assert _K.shape == (3, 2), 'The kernel matrix should have shape (N, M)'
assert np.isclose(_K[1, 0], 4.), 'For identical positions the kernel should equal sigma_f^2'
assert np.isclose(_K[0, 0], 4.*np.exp(-0.5)), 'Check the exponent: -(x1-x2)^2 / (2 ell^2)'
assert np.isclose(_K[0, 1], 4.*np.exp(-4.5)), 'Check the exponent: -(x1-x2)^2 / (2 ell^2)'
_KK = rbf_kernel(_X1, _X1, sigma_f=2., ell=1.)
assert np.allclose(_KK, _KK.T) and np.allclose(np.diag(_KK), 4.), 'K(X, X) should be symmetric with sigma_f^2 on the diagonal'
assert rbf_kernel(np.array([0., 1., 2.]), np.array([1., 3.]), sigma_f=2., ell=1.).shape == (3, 2), \
    'rbf_kernel should also accept one-dimensional arrays of positions'

In [ ]:
# Utility function for plotting a GP prediction with credible interval
def gp_plot(X_new, mu_new, cov_new, ax=None, credibility=0.95, show_samples=False, num_samples=3, rng=rng):
    '''
    Plot predictions of a GP regression model.

    Args:
        X_new: new input locations, array-like of length N_new
        mu_new: GP mean, array of shape (N_new, 1) or (N_new,)
        cov_new: GP covariance matrix, array of shape (N_new, N_new)
        ax: matplotlib axes (created if None)
        credibility: (float) credibility of the shaded band
        show_samples: (bool) whether to draw and plot samples from the GP
        num_samples: (int) number of samples
        rng: numpy random Generator
    '''
    if ax is None:
        fig, ax = plt.subplots(1, 1)
    X_new = np.asarray(X_new).ravel()
    mu_new = np.asarray(mu_new).ravel()
    std_new = np.sqrt(np.diag(cov_new))
    z = stats.norm.ppf(0.5 + credibility/2)
    ax.plot(X_new, mu_new, 'C0-', label='mean')
    ax.fill_between(X_new, mu_new - z*std_new, mu_new + z*std_new, color='C0', alpha=0.2,
                    label=f'{100*credibility:.0f}% credible band')
    if show_samples:
##################
# YOUR CODE HERE #
##################
    return ax

In [ ]:
# Plot the prior over functions
sigma_n = 0.2
sigma_f_0, ell_0 = 3., 5.
X_pred = np.arange(50, 91).reshape(-1, 1).astype(float)

##################
# YOUR CODE HERE #
##################


### (b) The posterior with fixed hyperparameters (1 point)

The `sklearn.gaussian_process` library provides a `GaussianProcessRegressor`
for implementing [GP regression models](http://scikit-learn.org/stable/modules/gaussian_process.html#gaussian-process-regression-gpr).
It can be configured with [pre-defined and user-defined kernels](http://scikit-learn.org/stable/modules/gaussian_process.html#gp-kernels)
that can be composed from several ingredients. The squared exponential kernel
is the `RBF` kernel in scikit-learn. It only has a `length_scale` parameter,
which corresponds to $\ell$ above. To have a $\sigma_f$ parameter as well, we
multiply the `RBF` kernel with a `ConstantKernel` (whose `constant_value` is
$\sigma_f^2$).

In this problem we include the **fixed** noise via the `alpha` argument of
`GaussianProcessRegressor`, which adds $\sigma_n^2$ to the diagonal of the
kernel matrix of the training data, i.e. it builds
$C(\mathbf{X}, \mathbf{X}) = K(\mathbf{X}, \mathbf{X}) + \sigma_n^2 \mathbb{1}$
as in (a).

**Latent function or new measurements?** For prediction points $\mathbf{X}_*$
there are two posterior distributions with the same mean but different
covariance: that of the *latent function* $f_*$, and that of *new
measurements* $y_* = f_* + \varepsilon_*$, whose covariance is larger by
$\sigma_n^2 \mathbb{1}$. Which one to use depends on the question. The first
answers "what is the smooth residual as a function of $N$", the second "where
would a new data point fall". In this problem we use the second, since we will
compare the band with actual data points in (d). `predict(return_cov=True)`
of a model with the noise in `alpha` returns the covariance of the latent
function, so $\sigma_n^2$ has to be added to its diagonal by hand.

**Tasks (see also Yata)**
* Set up the corresponding GP regression model using `sklearn`.
  **Important**: set `optimizer=None` and `normalize_y=False` in this task, so
  that the hyperparameters are kept fixed (the demonstration notebook uses
  `normalize_y=True`; do not copy that here).
* Fit this GP regression model, `gpr`, to the training data (`X_train`, `Y_train`)
  with the hyperparameters of the previous task.
* Make posterior predictions for $N \in [50, 90]$. Extract the mean vector and
  covariance matrix and plot the posterior prediction with `gp_plot`, showing
  5 samples. Add the training data to the plot.
  **Important**: include the argument `return_cov=True` in `predict()` to get
  the covariance matrix, and add `sigma_n**2` to its diagonal to obtain the
  predictive covariance for new measurements.
* Print the log marginal likelihood of the training data for these
  hyperparameters (`gpr.log_marginal_likelihood_value_`); you will compare it
  with the optimized value in (c).

In [ ]:
# GP regression with fixed hyperparameters using sklearn.
##################
# YOUR CODE HERE #
##################


### (c) Hyperparameter optimization and your own posterior prediction (1 point)

**Tasks (see also Yata)**
* Fit the hyperparameters using `sklearn`: set up the GP model with an
  optimizer (the default `optimizer="fmin_l_bfgs_b"` works fine; use a few restarts
  `n_restarts_optimizer`). Extract and print the optimized values
  $\sigma_{f,\mathrm{opt}}$ and $\ell_\mathrm{opt}$: after the fit, the
  optimized kernel is `gpr_opt.kernel_`, whose two factors are `k1` (the
  `ConstantKernel`, with `constant_value` $= \sigma_f^2$) and `k2` (the `RBF`,
  with `length_scale` $= \ell$), i.e. `gpr_opt.kernel_.k1.constant_value` and
  `gpr_opt.kernel_.k2.length_scale`. Compare the log marginal likelihood with
  the value from (b), and plot the posterior prediction for $N \in [50, 90]$
  with 5 samples.
* Write your own `gp_predict` function (see below) that computes the
  posterior mean and covariance for new measurements at $\mathbf{X}_*$,
  $$
  \boldsymbol{\mu}_* = K(\mathbf{X}_*, \mathbf{X})\, \left[ K(\mathbf{X}, \mathbf{X}) + \sigma_n^2 \mathbb{1} \right]^{-1} \mathbf{y}, \qquad
  \boldsymbol{\Sigma}_* = K(\mathbf{X}_*, \mathbf{X}_*) + \sigma_n^2 \mathbb{1} - K(\mathbf{X}_*, \mathbf{X})\, \left[ K(\mathbf{X}, \mathbf{X}) + \sigma_n^2 \mathbb{1} \right]^{-1} K(\mathbf{X}, \mathbf{X}_*),
  $$
  with the kernel matrices from `rbf_kernel`. Note where the noise appears:
  on the diagonal of the training covariance (the data are noisy), on the
  diagonal of the prediction covariance (we predict noisy measurements), but
  not in the cross-covariance $K(\mathbf{X}_*, \mathbf{X})$, since the noise
  of a new measurement is independent of the noise in the training data. Use
  `numpy.linalg.solve` (or a Cholesky decomposition) rather than an explicit
  matrix inverse.
* Check that, with the optimized hyperparameters, your function reproduces the
  mean vector and the covariance matrix (with $\sigma_n^2$ added to the
  diagonal) of the optimized `sklearn` model for the same prediction points.

In [ ]:
# Optimize the hyperparameters with sklearn. Store them in sigma_f_opt and ell_opt.
##################
# YOUR CODE HERE #
##################


In [ ]:
def gp_predict(X_new, X_train, Y_train, sigma_f, ell, sigma_n):
    '''
    Posterior mean and covariance of a GP regression model with an RBF kernel and white noise.

    Args:
        X_new: new input locations, shape (N_new, 1)
        X_train: training locations, shape (N_train, 1)
        Y_train: training targets, shape (N_train, 1)
        sigma_f, ell: kernel hyperparameters
        sigma_n: noise standard deviation (the returned covariance is that of new measurements,
            i.e. it includes sigma_n^2 on the diagonal)

    Returns:
        mu_new: array of shape (N_new, 1)
        cov_new: array of shape (N_new, N_new)
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
mu_own, cov_own = gp_predict(X_pred, X_train, Y_train, sigma_f_opt, ell_opt, sigma_n)
assert mu_own.shape == (len(X_pred), 1), 'mu_new should have shape (N_new, 1)'
assert cov_own.shape == (len(X_pred), len(X_pred)), 'cov_new should have shape (N_new, N_new)'
assert np.allclose(mu_own.ravel(), mu_opt.ravel(), atol=1e-6), 'The mean does not agree with the sklearn model'
assert np.allclose(cov_own, cov_opt, atol=1e-6), 'The covariance does not agree with the sklearn model (did you include the noise on the diagonal?)'
print('Your posterior agrees with sklearn.')

### (d) Prediction beyond the data (1 point)

**Tasks (see also Yata)**
* Use `gp_predict` with the optimized hyperparameters to predict the residual
  for $N = 50, \ldots, 110$. Plot the mean and the 95% credible band together
  with the training data and the five held-out nuclei ($N = 81, \ldots, 85$).
* How many of the five held-out points fall inside the 95% band? Where does
  the GP mean go for large $N$, and why? What does the growing width of the
  band express?
* The GP is a model too. What assumption, built into the kernel, is responsible
  for the failure (or success) you observe just above $N = 82$? Does the GP
  "know what it does not know"?

In [ ]:
# Predict far beyond the training data and compare with the held-out nuclei.
##################
# YOUR CODE HERE #
##################


**Your comments here**

****************************
** Write your answer here **
****************************
